# IceWave — ML Model v2 Split (Ecoregion)

Two separate Random Forest models split at the Cascade crest (~-121.5°W):

| Model | Ecoregion | Training Points | Source |
|-------|-----------|-----------------|--------|
| **West** | Willamette Valley / Puget Sound / Coast Range | 31 | merged occ_clean |
| **East** | Columbia Plateau / Basin & Range / Nevada | 17 | PBDB terrain CSV |

Top 25 targets from each → merged 50-target final list with ecoregion provenance.

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import rasterio.enums
import rasterio.transform
from pathlib import Path
from shapely.geometry import Point
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
import joblib

Path('../data/model').mkdir(parents=True, exist_ok=True)
Path('../outputs').mkdir(parents=True, exist_ok=True)

CASCADE_SPLIT = -121.5
FEATURES = ['elevation', 'slope', 'aspect', 'tri', 'lith_score']

TERRAIN_FILES = {
    'elevation' : '../data/terrain/dem_mosaic.tif',
    'slope'     : '../data/terrain/slope.tif',
    'aspect'    : '../data/terrain/aspect.tif',
    'tri'       : '../data/terrain/ruggedness.tif',
}

LITH_SCORE_MAP = {
    'Clay': 1.0, 'Silt': 1.0, 'Sand': 1.0, 'Gravel': 1.0,
    'Unconsolidated': 1.0, 'Alluvium': 1.0, 'Loess': 1.0,
    'Sandstone': 0.9, 'Mudstone': 0.9, 'Shale': 0.9,
    'Limestone': 0.7, 'Dolomite': 0.6,
    'Granite': 0.2, 'Basalt': 0.15, 'Andesite': 0.15,
    'Rhyolite': 0.15, 'Metamorphic': 0.1, 'Gneiss': 0.1,
}

def score_lith(lith_str):
    if pd.isna(lith_str):
        return 0.4
    for key, val in LITH_SCORE_MAP.items():
        if key.lower() in str(lith_str).lower():
            return val
    return 0.4

print('Setup complete.')

Setup complete.


## 1. Prepare Occurrence Datasets

In [3]:
# ── West: from merged enriched dataset ────────────────────────────────────
occ_merged = pd.read_csv('../data/merged/icewave_merged_occurrences.csv')
occ_terrain = pd.read_csv('../data/pbdb/icewave_occurrences_with_terrain.csv')

# Attach terrain to merged points
occ_terrain['_key'] = occ_terrain['latitude'].round(4).astype(str) + '_' + occ_terrain['longitude'].round(4).astype(str)
occ_merged['_key']  = occ_merged['latitude'].round(4).astype(str)  + '_' + occ_merged['longitude'].round(4).astype(str)

terrain_lookup = occ_terrain.set_index('_key')[['elevation','slope','aspect','tri']]
occ_merged = occ_merged.join(terrain_lookup, on='_key', how='left')
occ_merged['lith_score'] = occ_merged['lith_score'].fillna(0.5)
occ_merged = occ_merged.dropna(subset=FEATURES)

# Filter to study area and split
occ_merged = occ_merged[occ_merged['latitude'] > 40.0].copy()
occ_merged['ecoregion'] = np.where(occ_merged['longitude'] < CASCADE_SPLIT, 'west', 'east')

west = occ_merged[occ_merged['ecoregion'] == 'west'].copy()

# ── East: from PBDB terrain CSV (more complete) ───────────────────────────
occ_terrain['ecoregion'] = np.where(occ_terrain['longitude'] < CASCADE_SPLIT, 'west', 'east')
east_raw = occ_terrain[
    (occ_terrain['ecoregion'] == 'east') &
    ~((occ_terrain['latitude'] < 37) & (occ_terrain['longitude'] < -118.5))
].copy()
lith_lookup = occ_merged.drop_duplicates('_key').set_index('_key')['lith_score']
east_raw['lith_score'] = east_raw['_key'].map(lith_lookup).fillna(0.4)
east = east_raw.drop_duplicates(subset=['latitude','longitude']).reset_index(drop=True)

print(f'West occurrence points : {len(west)}')
print(f'  Lat: {west.latitude.min():.1f} – {west.latitude.max():.1f}')
print(f'  Lon: {west.longitude.min():.1f} – {west.longitude.max():.1f}')
print(f'\nEast occurrence points : {len(east)}')
print(f'  Lat: {east.latitude.min():.1f} – {east.latitude.max():.1f}')
print(f'  Lon: {east.longitude.min():.1f} – {east.longitude.max():.1f}')

West occurrence points : 35
  Lat: 42.1 – 48.6
  Lon: -123.5 – -122.8

East occurrence points : 17
  Lat: 36.1 – 47.5
  Lon: -120.8 – -114.0


## 2. Generate Background Points (by Ecoregion)

In [4]:
def generate_background(n_presence, ecoregion, lon_bounds, ratio=8, seed=42):
    """Sample background points from DEM within ecoregion lon bounds."""
    N = n_presence * ratio
    np.random.seed(seed)

    with rasterio.open(TERRAIN_FILES['elevation']) as src:
        bounds = src.bounds
        lon_min = max(bounds.left,  lon_bounds[0])
        lon_max = min(bounds.right, lon_bounds[1])
        bg_lons = np.random.uniform(lon_min, lon_max, N * 3)
        bg_lats = np.random.uniform(bounds.bottom, bounds.top, N * 3)
        coords  = list(zip(bg_lons, bg_lats))
        elev_vals = np.array([v[0] for v in src.sample(coords)])

    valid = (elev_vals > -500) & (elev_vals < 5000) & (~np.isnan(elev_vals))
    bg_lons = bg_lons[valid][:N]
    bg_lats = bg_lats[valid][:N]

    bg = pd.DataFrame({'longitude': bg_lons, 'latitude': bg_lats})
    bg_coords = list(zip(bg['longitude'], bg['latitude']))

    sources = {k: rasterio.open(v) for k, v in TERRAIN_FILES.items()}
    for feat, src in sources.items():
        bg[feat] = np.array([v[0] for v in src.sample(bg_coords)])
    for src in sources.values():
        src.close()

    bg['lith_score'] = 0.4
    bg = bg.dropna(subset=FEATURES).reset_index(drop=True)
    print(f'{ecoregion} background: {len(bg)} points ({ratio}:1 ratio)')
    return bg

bg_west = generate_background(len(west), 'west', lon_bounds=(-125.0, CASCADE_SPLIT), ratio=8)
bg_east = generate_background(len(east), 'east', lon_bounds=(CASCADE_SPLIT, -110.0), ratio=5)

west background: 154 points (8:1 ratio)
east background: 49 points (5:1 ratio)


## 3. Train West Model

In [5]:
def build_training(presence_df, background_df):
    presence_df = presence_df.copy()
    background_df = background_df.copy()
    presence_df['presence'] = 1
    background_df['presence'] = 0
    df = pd.concat(
        [presence_df[FEATURES + ['presence', 'latitude', 'longitude']],
         background_df[FEATURES + ['presence', 'latitude', 'longitude']]],
        ignore_index=True
    ).dropna(subset=FEATURES)
    return df

def train_rf(train_df, label, n_estimators=500, min_samples_leaf=3):
    X = train_df[FEATURES].values
    y = train_df['presence'].values
    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        min_samples_leaf=min_samples_leaf,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    auc_scores = cross_val_score(rf, X, y, cv=cv, scoring='roc_auc')
    rf.fit(X, y)
    fi = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
    print(f'\n── {label} Model ──────────────────────────')
    print(f'  Training points  : {y.sum()} presence / {(y==0).sum()} background')
    print(f'  CV AUC           : {auc_scores.mean():.3f} ± {auc_scores.std():.3f}')
    print(f'  Feature importances:')
    for feat, imp in fi.items():
        print(f'    {feat:<15} {imp:.3f}')
    return rf, auc_scores

train_west = build_training(west, bg_west)
rf_west, auc_west = train_rf(train_west, 'WEST (Willamette/Puget)')
joblib.dump(rf_west, '../data/model/icewave_rf_west.joblib')
print('  Saved: ../data/model/icewave_rf_west.joblib')


── WEST (Willamette/Puget) Model ──────────────────────────
  Training points  : 35 presence / 154 background
  CV AUC           : 0.890 ± 0.105
  Feature importances:
    lith_score      0.373
    elevation       0.249
    tri             0.178
    slope           0.115
    aspect          0.086
  Saved: ../data/model/icewave_rf_west.joblib


## 4. Train East Model

In [6]:
train_east = build_training(east, bg_east)
rf_east, auc_east = train_rf(train_east, 'EAST (Columbia Plateau/Basin & Range)',
                              min_samples_leaf=2)  # smaller leaf size for small dataset
joblib.dump(rf_east, '../data/model/icewave_rf_east.joblib')
print('  Saved: ../data/model/icewave_rf_east.joblib')


── EAST (Columbia Plateau/Basin & Range) Model ──────────────────────────
  Training points  : 17 presence / 49 background
  CV AUC           : 0.566 ± 0.121
  Feature importances:
    aspect          0.270
    lith_score      0.189
    tri             0.187
    slope           0.180
    elevation       0.174
  Saved: ../data/model/icewave_rf_east.joblib


## 5. Generate Candidate Grids (by Ecoregion)

In [14]:
final = pd.concat([top_west, top_east], ignore_index=True)

# Normalize composite within each ecoregion then re-rank globally
for eco in ['west', 'east']:
    mask = final['ecoregion'] == eco
    vals = final.loc[mask, 'composite']
    final.loc[mask, 'composite_norm'] = (vals - vals.min()) / (vals.max() - vals.min())

final = final.sort_values('composite_norm', ascending=False).reset_index(drop=True)
final.index += 1
final.index.name = 'rank'

# Add confidence flag
final['confidence'] = np.where(final['ecoregion'] == 'west', 'high', 'low')

print(f'Final target list: {len(final)} sites')
print(f'  West targets (HIGH confidence): {(final.ecoregion=="west").sum()}')
print(f'  East targets (LOW confidence) : {(final.ecoregion=="east").sum()}')
print()
print('Top 10:')
print(final[['ecoregion','confidence','latitude','longitude','prob','lith_score',
             'composite_norm','elevation']].head(10).round(4).to_string())

final.to_csv('../data/model/icewave_v2_split_top50.csv')
print('\nSaved: ../data/model/icewave_v2_split_top50.csv')

Final target list: 42 sites
  West targets (HIGH confidence): 17
  East targets (LOW confidence) : 25

Top 10:
     ecoregion confidence  latitude  longitude    prob  lith_score  composite_norm   elevation
rank                                                                                          
1         west       high   45.1753  -122.8419  0.9954         1.0          1.0000   56.367199
2         east        low   46.9114  -120.3281  0.8907         1.0          1.0000  605.877197
3         west       high   48.5781  -122.2586  0.9906         1.0          0.9863   52.823799
4         west       high   45.8281  -122.6058  0.9897         1.0          0.9837   59.448200
5         west       high   47.4669  -122.0919  0.9896         1.0          0.9834   52.856400
6         west       high   48.1336  -123.5364  0.9863         1.0          0.9739   56.379398
7         west       high   48.4947  -122.8975  0.9734         1.0          0.9370   67.698502
8         east        low   46.078

In [15]:
import simplekml
import gpxpy
import gpxpy.gpx

WEST_COLOR = simplekml.Color.cyan    # cyan  = high confidence
EAST_COLOR = simplekml.Color.yellow  # yellow = low confidence

# ── KMZ ───────────────────────────────────────────────────────────────────
kml = simplekml.Kml()
fol_west = kml.newfolder(name='West — Willamette/Puget (HIGH confidence)')
fol_east = kml.newfolder(name='East — Columbia Plateau/Basin & Range (LOW confidence)')

for rank, row in final.iterrows():
    folder = fol_west if row['ecoregion'] == 'west' else fol_east
    color  = WEST_COLOR if row['ecoregion'] == 'west' else EAST_COLOR
    conf   = 'HIGH' if row['ecoregion'] == 'west' else 'LOW — east model AUC 0.566'
    desc = (
        f'Global Rank    : {rank}\n'
        f'Ecoregion      : {row["ecoregion"].upper()}\n'
        f'Confidence     : {conf}\n'
        f'ML Probability : {row["prob"]:.3f}\n'
        f'Lith Score     : {row["lith_score"]:.2f}\n'
        f'Composite Score: {row["composite_norm"]:.3f}\n'
        f'Elevation      : {row["elevation"]:.0f} m\n'
        f'Slope          : {row["slope"]:.1f}°\n'
        f'TRI            : {row["tri"]:.2f}'
    )
    pt = folder.newpoint(
        name=f'IW-{row["ecoregion"][0].upper()}{rank:02d}',
        coords=[(row['longitude'], row['latitude'])],
        description=desc
    )
    pt.style.iconstyle.color = color
    if rank <= 5:
        pt.style.iconstyle.scale = 1.5
    elif rank <= 10:
        pt.style.iconstyle.scale = 1.2
    else:
        pt.style.iconstyle.scale = 1.0

kml.savekmz('../outputs/icewave_v2_split_targets.kmz')
print('Saved: ../outputs/icewave_v2_split_targets.kmz')

# ── GPX ───────────────────────────────────────────────────────────────────
gpx = gpxpy.gpx.GPX()
for rank, row in final.iterrows():
    conf = 'HIGH' if row['ecoregion'] == 'west' else 'LOW'
    wpt = gpxpy.gpx.GPXWaypoint(
        latitude=row['latitude'],
        longitude=row['longitude'],
        elevation=row['elevation'],
        name=f'IW-{row["ecoregion"][0].upper()}{rank:02d}',
        comment=f'{row["ecoregion"].upper()} {conf} Score={row["composite_norm"]:.3f} Lith={row["lith_score"]:.2f}'
    )
    gpx.waypoints.append(wpt)

with open('../outputs/icewave_v2_split_targets.gpx', 'w') as f:
    f.write(gpx.to_xml())
print('Saved: ../outputs/icewave_v2_split_targets.gpx')

Saved: ../outputs/icewave_v2_split_targets.kmz
Saved: ../outputs/icewave_v2_split_targets.gpx


## 6. Predict & Extract Top 25 per Ecoregion

In [16]:
print('=' * 60)
print('  IceWave ML Model v2 Split — Results Summary')
print('=' * 60)
print(f'  Cascade split longitude : {CASCADE_SPLIT}°W')
print()
print(f'  WEST model (Willamette/Puget/Coast Range)')
print(f'    Training points : {len(west)} presence')
print(f'    CV AUC          : {auc_west.mean():.3f} ± {auc_west.std():.3f}')
print(f'    Confidence      : HIGH')
print()
print(f'  EAST model (Columbia Plateau/Basin & Range)')
print(f'    Training points : {len(east)} presence')
print(f'    CV AUC          : {auc_east.mean():.3f} ± {auc_east.std():.3f}')
print(f'    Confidence      : LOW — insufficient training data')
print()
print(f'  v1 baseline AUC  : 0.853 ± 0.063')
print()
print('  Top 10 Targets (global rank):')
for rank, row in final.head(10).iterrows():
    eco  = row['ecoregion'].upper()
    conf = '★' if row['confidence'] == 'high' else '○'
    print(f'    {conf} #{rank:02d} [{eco}]  {row["latitude"]:.4f}°N  '
          f'{row["longitude"]:.4f}°W  score={row["composite_norm"]:.3f}')
print()
print('  ★ = HIGH confidence (west, AUC 0.890)')
print('  ○ = LOW confidence  (east, AUC 0.566, n=17 training points)')
print()
print('  Outputs:')
print('    ../data/model/icewave_v2_split_top50.csv')
print('    ../data/model/icewave_rf_west.joblib')
print('    ../data/model/icewave_rf_east.joblib')
print('    ../outputs/icewave_v2_split_targets.kmz')
print('    ../outputs/icewave_v2_split_targets.gpx')
print('=' * 60)

  IceWave ML Model v2 Split — Results Summary
  Cascade split longitude : -121.5°W

  WEST model (Willamette/Puget/Coast Range)
    Training points : 35 presence
    CV AUC          : 0.890 ± 0.105
    Confidence      : HIGH

  EAST model (Columbia Plateau/Basin & Range)
    Training points : 17 presence
    CV AUC          : 0.566 ± 0.121
    Confidence      : LOW — insufficient training data

  v1 baseline AUC  : 0.853 ± 0.063

  Top 10 Targets (global rank):
    ★ #01 [WEST]  45.1753°N  -122.8419°W  score=1.000
    ○ #02 [EAST]  46.9114°N  -120.3281°W  score=1.000
    ★ #03 [WEST]  48.5781°N  -122.2586°W  score=0.986
    ★ #04 [WEST]  45.8281°N  -122.6058°W  score=0.984
    ★ #05 [WEST]  47.4669°N  -122.0919°W  score=0.983
    ★ #06 [WEST]  48.1336°N  -123.5364°W  score=0.974
    ★ #07 [WEST]  48.4947°N  -122.8975°W  score=0.937
    ○ #08 [EAST]  46.0781°N  -118.2308°W  score=0.936
    ★ #09 [WEST]  47.6475°N  -121.8975°W  score=0.925
    ★ #10 [WEST]  45.3836°N  -122.4947°W  score=

In [17]:
def extract_top_n(cand, rf, n=25, label='', bg_lith=0.4):
    """Predict, score composite, thin spatially, return top N."""
    cand = cand.copy()
    cand['prob'] = rf.predict_proba(cand[FEATURES].values)[:, 1]
    cand['composite'] = (0.80 * cand['prob']) + (0.20 * cand['lith_score'])
    cand['ecoregion'] = label

    # Filter to Quaternary deposits
    if has_quat:
        gdf = gpd.GeoDataFrame(
            cand, geometry=gpd.points_from_xy(cand['longitude'], cand['latitude']),
            crs='EPSG:4326'
        )
        in_quat = gpd.sjoin(gdf, quat_dep[['geometry']], how='inner', predicate='within')
        pool = cand.loc[in_quat.index.unique()]
        print(f'{label}: {len(pool):,} candidates in Quaternary deposits')
    else:
        pool = cand

    # Spatial thinning — best per ~50km cell
    pool = pool.copy()
    pool['cell'] = (pool['latitude'].round(0).astype(str) + '_' +
                    pool['longitude'].round(0).astype(str))
    thinned = pool.sort_values('composite', ascending=False).drop_duplicates('cell')

    top = thinned.nlargest(n, 'composite').reset_index(drop=True)
    top.index += 1
    top.index.name = 'local_rank'

    print(f'{label}: top {n} extracted')
    print(f'  Prob range : {top["prob"].min():.3f} – {top["prob"].max():.3f}')
    print(f'  Lith mean  : {top["lith_score"].mean():.3f}')
    return top

top_west = extract_top_n(cand_west, rf_west, n=25, label='west')
top_east = extract_top_n(cand_east, rf_east, n=25, label='east')

west: 10,728 candidates in Quaternary deposits
west: top 25 extracted
  Prob range : 0.646 – 0.995
  Lith mean  : 1.000
east: 32,012 candidates in Quaternary deposits
east: top 25 extracted
  Prob range : 0.776 – 0.891
  Lith mean  : 1.000


## 7. Merge into Final 50-Target List

In [18]:
final = pd.concat([top_west, top_east], ignore_index=True)

# Normalize composite within each ecoregion then re-rank globally
for eco in ['west', 'east']:
    mask = final['ecoregion'] == eco
    vals = final.loc[mask, 'composite']
    final.loc[mask, 'composite_norm'] = (vals - vals.min()) / (vals.max() - vals.min())

final = final.sort_values('composite_norm', ascending=False).reset_index(drop=True)
final.index += 1
final.index.name = 'rank'

print(f'Final target list: {len(final)} sites')
print(f'  West targets: {(final.ecoregion=="west").sum()}')
print(f'  East targets: {(final.ecoregion=="east").sum()}')
print()
print('Top 10:')
print(final[['ecoregion','latitude','longitude','prob','lith_score',
             'composite_norm','elevation']].head(10).round(4).to_string())

final.to_csv('../data/model/icewave_v2_split_top50.csv')
print('\nSaved: ../data/model/icewave_v2_split_top50.csv')

Final target list: 42 sites
  West targets: 17
  East targets: 25

Top 10:
     ecoregion  latitude  longitude    prob  lith_score  composite_norm   elevation
rank                                                                               
1         west   45.1753  -122.8419  0.9954         1.0          1.0000   56.367199
2         east   46.9114  -120.3281  0.8907         1.0          1.0000  605.877197
3         west   48.5781  -122.2586  0.9906         1.0          0.9863   52.823799
4         west   45.8281  -122.6058  0.9897         1.0          0.9837   59.448200
5         west   47.4669  -122.0919  0.9896         1.0          0.9834   52.856400
6         west   48.1336  -123.5364  0.9863         1.0          0.9739   56.379398
7         west   48.4947  -122.8975  0.9734         1.0          0.9370   67.698502
8         east   46.0781  -118.2308  0.8834         1.0          0.9363  480.480591
9         west   47.6475  -121.8975  0.9691         1.0          0.9246   35.737999
1

## 8. Export KMZ & GPX

In [19]:
import simplekml
import gpxpy
import gpxpy.gpx

WEST_COLOR = simplekml.Color.cyan
EAST_COLOR = simplekml.Color.orange

# ── KMZ ───────────────────────────────────────────────────────────────────
kml = simplekml.Kml()
fol_west = kml.newfolder(name='West — Willamette/Puget')
fol_east = kml.newfolder(name='East — Columbia Plateau/Basin & Range')

for rank, row in final.iterrows():
    folder = fol_west if row['ecoregion'] == 'west' else fol_east
    color  = WEST_COLOR if row['ecoregion'] == 'west' else EAST_COLOR
    desc = (
        f'Global Rank    : {rank}\n'
        f'Ecoregion      : {row["ecoregion"].upper()}\n'
        f'ML Probability : {row["prob"]:.3f}\n'
        f'Lith Score     : {row["lith_score"]:.2f}\n'
        f'Composite Score: {row["composite_norm"]:.3f}\n'
        f'Elevation      : {row["elevation"]:.0f} m\n'
        f'Slope          : {row["slope"]:.1f}°\n'
        f'TRI            : {row["tri"]:.2f}'
    )
    pt = folder.newpoint(
        name=f'IW-{row["ecoregion"][0].upper()}{rank:02d}',
        coords=[(row['longitude'], row['latitude'])],
        description=desc
    )
    pt.style.iconstyle.color = color
    if rank <= 5:
        pt.style.iconstyle.scale = 1.5
    elif rank <= 10:
        pt.style.iconstyle.scale = 1.2
    else:
        pt.style.iconstyle.scale = 1.0

kml.savekmz('../outputs/icewave_v2_split_targets.kmz')
print('Saved: ../outputs/icewave_v2_split_targets.kmz')

# ── GPX ───────────────────────────────────────────────────────────────────
gpx = gpxpy.gpx.GPX()
for rank, row in final.iterrows():
    wpt = gpxpy.gpx.GPXWaypoint(
        latitude=row['latitude'],
        longitude=row['longitude'],
        elevation=row['elevation'],
        name=f'IW-{row["ecoregion"][0].upper()}{rank:02d}',
        comment=f'{row["ecoregion"].upper()} Score={row["composite_norm"]:.3f} Lith={row["lith_score"]:.2f}'
    )
    gpx.waypoints.append(wpt)

with open('../outputs/icewave_v2_split_targets.gpx', 'w') as f:
    f.write(gpx.to_xml())
print('Saved: ../outputs/icewave_v2_split_targets.gpx')

Saved: ../outputs/icewave_v2_split_targets.kmz
Saved: ../outputs/icewave_v2_split_targets.gpx


## 9. Summary

In [20]:
print('=' * 60)
print('  IceWave ML Model v2 Split — Results Summary')
print('=' * 60)
print(f'  Cascade split longitude : {CASCADE_SPLIT}°W')
print()
print(f'  WEST model (Willamette/Puget/Coast Range)')
print(f'    Training points : {len(west)} presence')
print(f'    CV AUC          : {auc_west.mean():.3f} ± {auc_west.std():.3f}')
print()
print(f'  EAST model (Columbia Plateau/Basin & Range)')
print(f'    Training points : {len(east)} presence')
print(f'    CV AUC          : {auc_east.mean():.3f} ± {auc_east.std():.3f}')
print()
print(f'  v1 baseline AUC  : 0.853 ± 0.063')
print()
print('  Top 10 Targets (global rank):')
for rank, row in final.head(10).iterrows():
    eco = row['ecoregion'].upper()
    print(f'    #{rank:02d} [{eco}]  {row["latitude"]:.4f}°N  '
          f'{row["longitude"]:.4f}°W  score={row["composite_norm"]:.3f}')
print()
print('  Outputs:')
print('    ../data/model/icewave_v2_split_top50.csv')
print('    ../data/model/icewave_rf_west.joblib')
print('    ../data/model/icewave_rf_east.joblib')
print('    ../outputs/icewave_v2_split_targets.kmz')
print('    ../outputs/icewave_v2_split_targets.gpx')
print('=' * 60)

  IceWave ML Model v2 Split — Results Summary
  Cascade split longitude : -121.5°W

  WEST model (Willamette/Puget/Coast Range)
    Training points : 35 presence
    CV AUC          : 0.890 ± 0.105

  EAST model (Columbia Plateau/Basin & Range)
    Training points : 17 presence
    CV AUC          : 0.566 ± 0.121

  v1 baseline AUC  : 0.853 ± 0.063

  Top 10 Targets (global rank):
    #01 [WEST]  45.1753°N  -122.8419°W  score=1.000
    #02 [EAST]  46.9114°N  -120.3281°W  score=1.000
    #03 [WEST]  48.5781°N  -122.2586°W  score=0.986
    #04 [WEST]  45.8281°N  -122.6058°W  score=0.984
    #05 [WEST]  47.4669°N  -122.0919°W  score=0.983
    #06 [WEST]  48.1336°N  -123.5364°W  score=0.974
    #07 [WEST]  48.4947°N  -122.8975°W  score=0.937
    #08 [EAST]  46.0781°N  -118.2308°W  score=0.936
    #09 [WEST]  47.6475°N  -121.8975°W  score=0.925
    #10 [WEST]  45.3836°N  -122.4947°W  score=0.903

  Outputs:
    ../data/model/icewave_v2_split_top50.csv
    ../data/model/icewave_rf_west.jobli